Merge the staging & target table on scd1 merge

In [0]:
from delta.tables import DeltaTable

# Define the target table and catalog/schema
staging_table = "test.default.staging_order_data"
target_table = "test.default.target_order_data"

# Read the staging data
staging_df = spark.read.table(staging_table)


# Check if the target table exists
if DeltaTable.isDeltaTable(spark, f"delta.`{target_table}`"):
    # Table exists, perform the merge
    target_delta = DeltaTable.forName(spark, target_table)

    # Your SCD1 Merge logic here
    target_delta.alias("target").merge(
        staging_df.alias("source"),
        "target.order_num = source.order_num"
    ).whenMatchedUpdate(
        condition="target.last_update_timestamp < source.last_update_timestamp",  # Update when the source is newer
        set={
            "target.tracking_num": "source.tracking_num",
            "target.pck_recieved_date": "source.pck_recieved_date",
            "target.package_deliver_date": "source.package_deliver_date",
            "target.status": "source.status",
            "target.address": "source.address",
            "target.last_update_timestamp": "source.last_update_timestamp"
        }
    ).whenNotMatchedInsert(
        values={
            "order_num": "source.order_num",
            "tracking_num": "source.tracking_num",
            "pck_recieved_date": "source.pck_recieved_date",
            "package_deliver_date": "source.package_deliver_date",
            "status": "source.status",
            "address": "source.address",
            "last_update_timestamp": "source.last_update_timestamp"
        }
    ).execute()
else:
    # If table doesn't exist, create it by writing the staging data
    staging_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
    print(f"Target table {target_table} created and data written.")


Now truncate the staging table

In [0]:
%sql 
truncate TABLE test.default.staging_order_data;